## Imports

In [1]:
# general
import random
import numpy as np
import pandas as pd
from collections import defaultdict
import random
import json

## Data handling


In [2]:
def tokenize(txtFile):
    stop_sentence_tokens = [".", "!", "?"]
    interrupt_sentence_tokens = [",", ";", ":"]

    words = ["<s>"]
    word = []
    for character in txtFile.read().lower():
        if character == "\n":
            if word:
                words.append("".join(word))
                words.append("</s>")
                words.append("<s>")            
                word = []
            continue    
        if character == " ":
            if word:
                words.append("".join(word))
                word = []
            continue
        if character in stop_sentence_tokens:
            if word:
                words.append("".join(word))
                words.append(character)
                words.append("</s>")
                words.append("<s>")
                word = []
            continue
        if character in interrupt_sentence_tokens:
            if word:
                words.append("".join(word))
                words.append(character)
                word = []
            continue
        word.append(character)
    return words

def vocabulary(words):
    return list(set(words))

def sentenceinizer(words):
    sentences = []
    sentence = []
    for word in words:
        sentence.append(word)
        if word == "</s>":
            sentences.append(sentence)
            sentence = []
    return sentences

def to_ngram_data(sentences, n=3):
    temp_sentences = [list(sentence) for sentence in sentences]
    for sentence in temp_sentences:
        for i in range(n):  
            sentence.insert(0, "<s>")
    return temp_sentences

In [3]:
txtFile = open("data/lotr.txt")
words = tokenize(txtFile)
vocab = vocabulary(words)
sentences = sentenceinizer(words)
trigram_sentences = to_ngram_data(sentences)

## BiGram

In [4]:
def compute_bigram_frequencies(sentences):
    bigram_frequencies = {}

    for sentence in sentences:
        for w1, w2 in zip(sentence, sentence[1:]):
            
            if w1 not in bigram_frequencies:
                bigram_frequencies[w1] = {}
            
            bigram_frequencies[w1][w2] = \
                bigram_frequencies[w1].get(w2, 0) + 1

    return bigram_frequencies

def bigram_frequencies_analysis(bigram_frequencies, top_n=10):
    total_bigrams = 0
    unique_bigrams = 0
    flat_bigrams = []

    for w1, next_words in bigram_frequencies.items():
        for w2, count in next_words.items():
            total_bigrams += count
            unique_bigrams += 1
            flat_bigrams.append(((w1, w2), count))

    sorted_bigrams = sorted(
        flat_bigrams,
        key=lambda x: x[1],
        reverse=True
    )

    print(f"Total bigrams: {total_bigrams}")
    print(f"Unique bigrams: {unique_bigrams}\n")

    print(f"Top {top_n} most frequent bigrams:")
    for (w1, w2), count in sorted_bigrams[:top_n]:
        print(f"({w1}, {w2}) -> {count}")


def bigram_generate_next_word(next_words_dict):
    words = list(next_words_dict.keys())
    weights = list(next_words_dict.values())
    return random.choices(words, weights=weights)[0]

def bigram_inference(bigram_frequencies, current_word="<s>"):
    result = [current_word]
    while current_word != "</s>":
        next_words = bigram_frequencies[current_word]
        current_word = bigram_generate_next_word(next_words)
        result.append(current_word)
    return result

def word_list_print(words):
    result = ''
    for word in words:
        if word in ["<s>", "</s>"]:
            continue

        if word in [".", ",", "!", "?", ":", ";"]:
            result = result.rstrip()
            result += word + " "
        else:
            result += word + " "
    result = result.strip()
    if result:
        result = result[0].upper() + result[1:]

    print(result)

In [5]:
bigram_frequencies = compute_bigram_frequencies(sentences)
#bigram_frequencies_analysis(bigram_frequencies)
sentence = bigram_inference(bigram_frequencies)
word_list_print(sentence)

We can talk of water they are no more tonight i return.


## Trigram

In [6]:
def compute_trigram_frequencies(sentences):
    trigram_frequencies = {}

    for sentence in sentences:
        for w1, w2, w3 in zip(sentence, sentence[1:], sentence[2:]):
            key = (w1, w2)

            if key not in trigram_frequencies:
                trigram_frequencies[key] = {}

            trigram_frequencies[key][w3] = \
                trigram_frequencies[key].get(w3, 0) + 1

    return trigram_frequencies

def trigram_frequencies_analysis(trigram_frequencies, top_n=10):
    total_trigrams = 0
    unique_trigrams = 0
    flat_trigrams = []

    for (w1, w2), next_words in trigram_frequencies.items():
        for w3, count in next_words.items():
            total_trigrams += count
            unique_trigrams += 1
            flat_trigrams.append(((w1, w2, w3), count))

    sorted_trigrams = sorted(
        flat_trigrams,
        key=lambda x: x[1],
        reverse=True
    )

    print(f"Total trigrams: {total_trigrams}")
    print(f"Unique trigrams: {unique_trigrams}\n")

    print(f"Top {top_n} most frequent trigrams:")
    for (w1, w2, w3), count in sorted_trigrams[:top_n]:
        print(f"({w1}, {w2}, {w3}) -> {count}")

def trigram_generate_next_word(next_words_dict):
    words = list(next_words_dict.keys())
    weights = list(next_words_dict.values())
    return random.choices(words, weights=weights)[0]

def trigram_inference(trigram_frequencies, current_words=("<s>", "<s>")):
    result = list(current_words)

    while True:
        key = (current_words[0], current_words[1])

        if key not in trigram_frequencies:
            break

        next_words = trigram_frequencies[key]
        next_word = trigram_generate_next_word(next_words)

        result.append(next_word)

        if next_word == "</s>":
            break

        current_words = (current_words[1], next_word)

    return result

In [7]:
trigram_frequencies = compute_trigram_frequencies(trigram_sentences)
trigram_frequencies_analysis(trigram_frequencies)
for i in range(10):
    sentence = trigram_inference(trigram_frequencies)
    word_list_print(sentence)

Total trigrams: 793219
Unique trigrams: 422505

Top 10 most frequent trigrams:
(<s>, <s>, <s>) -> 83090
(<s>, <s>, ') -> 6374
(<s>, <s>, the) -> 2139
(,, ', said) -> 1910
(<s>, <s>, but) -> 1735
(<s>, <s>, he) -> 1640
(<s>, <s>, i) -> 1358
(<s>, <s>, ") -> 1138
(<s>, <s>, and) -> 1134
(<s>, <s>, it) -> 1070
They were quarrelling about, and then he threw himself upon the first time since they left the shadow they hid themselves again.
' he said, "and very popular, for it.
'it seems a long slow sleep of death.
'he's often away from the shire, leave the halls within must be made a good deal of creeping and crawling over the edge of the tindrock.
' `then if the air was still afraid, ' said frodo.
Now the watch-towers, which we wish to drown in the leading company rode away with the fitful moonlight.
I don't see as well yourselves in his chair.
There was one swarthy bree-lander, who would you commit your promise to come and let it do at all, swift as any queen; but music turns her head.
The

## NGram

In [8]:
def compute_ngram_frequencies(sentences, n):
    ngram_frequencies = {}

    for sentence in sentences:
        for ngram in zip(*[sentence[i:] for i in range(n)]):
            key = ngram[:-1]     # first n-1 words
            next_word = ngram[-1] # last word

            if key not in ngram_frequencies:
                ngram_frequencies[key] = {}

            ngram_frequencies[key][next_word] = \
                ngram_frequencies[key].get(next_word, 0) + 1

    return ngram_frequencies

def ngram_frequencies_analysis(ngram_frequencies, top_n=10):
    total_ngrams = 0
    unique_ngrams = 0
    flat_ngrams = []

    for key, next_words in ngram_frequencies.items():
        for next_word, count in next_words.items():
            total_ngrams += count
            unique_ngrams += 1
            flat_ngrams.append(((*key, next_word), count))

    sorted_ngrams = sorted(
        flat_ngrams,
        key=lambda x: x[1],
        reverse=True
    )

    print(f"Total n-grams: {total_ngrams}")
    print(f"Unique n-grams: {unique_ngrams}\n")

    print(f"Top {top_n} most frequent n-grams:")
    for ngram, count in sorted_ngrams[:top_n]:
        print(f"{ngram} -> {count}")

def ngram_generate_next_word(next_words_dict):
    words = list(next_words_dict.keys())
    weights = list(next_words_dict.values())
    return random.choices(words, weights=weights)[0]

def ngram_inference(ngram_frequencies, n, current_words=None):
    if current_words is None:
        current_words = ("<s>",) * (n - 1)

    result = list(current_words)

    while True:
        key = tuple(current_words)

        if key not in ngram_frequencies:
            break

        next_words = ngram_frequencies[key]
        next_word = ngram_generate_next_word(next_words)

        result.append(next_word)

        if next_word == "</s>":
            break

        current_words = current_words[1:] + (next_word,)

    return result

In [9]:
n = 10
ngram_sentences = to_ngram_data(sentences, n)
ngram_frequencies = compute_ngram_frequencies(ngram_sentences, n)
ngram_frequencies_analysis(ngram_frequencies)
for i in range(10):
    sentence = ngram_inference(ngram_frequencies, n)
    word_list_print(sentence)

Total n-grams: 793219
Unique n-grams: 626389

Top 10 most frequent n-grams:
('<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>') -> 83090
('<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', "'") -> 6374
('<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', 'the') -> 2139
('<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', 'but') -> 1735
('<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', 'he') -> 1640
('<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', 'i') -> 1358
('<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '"') -> 1138
('<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', 'and') -> 1134
('<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', 'it') -> 1070
('<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', 'they') -> 1017
" "they may indeed, " said thorin.
Peregrin, ' he said.
It was a misty, moisty morning when we climbed down and looked round again, and nobo

## Save Model

In [10]:
def save_ngram_model(ngram_frequencies, filename):
    serializable = {}

    for key, next_words in ngram_frequencies.items():
        # Convert tuple key → string
        key_str = " ".join(key)
        serializable[key_str] = next_words

    with open(filename, "w") as f:
        json.dump(serializable, f, indent=4)

def load_ngram_model(filename):
    with open(filename, "r") as f:
        data = json.load(f)

    ngram_frequencies = {}

    for key_str, next_words in data.items():
        key = tuple(key_str.split())
        ngram_frequencies[key] = next_words

    return ngram_frequencies

In [11]:
save_ngram_model(ngram_frequencies, "model.json")
ngram_frequencies = load_ngram_model("model.json")

## Improvements

Smoothing - Kneser-Ney

Pruning

Backoff models